In [ ]:
import os, sys

HOME = "/home/j-j14d208"
CODE_DIR = f"{HOME}/2_ml_pipeline"
DATA_DIR = f"{HOME}/kospi200-project"
os.environ["BASE_PATH"] = DATA_DIR
os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import logging
import utils.logger as _log_mod
def setup_logger(name):
    logger = logging.getLogger(name)
    if logger.handlers: return logger
    logger.setLevel(logging.DEBUG)
    h = logging.StreamHandler(sys.stdout)
    h.setLevel(logging.INFO)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s", "%H:%M:%S"))
    logger.addHandler(h)
    return logger
_log_mod.setup_logger = setup_logger
print(f"CWD: {os.getcwd()}")

In [ ]:
!pip install arch --quiet

In [ ]:
"""GARCH Walk-Forward with Checkpoint Saving
각 윈도우별 종목별 GARCH(1,1) 변동성 예측을 저장.
GARCH는 종목별 모델이므로 체크포인트 = 윈도우별 예측 결과 DataFrame.
모델 자체 저장보다 예측값 저장이 실용적 (GARCH는 피팅이 빠름).
"""
import warnings, json, joblib
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from arch import arch_model

warnings.filterwarnings("ignore")
print("GARCH Walk-Forward 준비 완료")

In [ ]:
# ===== 설정 =====
BASE_PATH = Path(os.environ.get("BASE_PATH"))
RAW_OHLCV_PATH = BASE_PATH / "raw" / "ohlcv"
TFT_FEATURE_PATH = BASE_PATH / "processed" / "tft_features" / "tft_features.parquet"
MODEL_SAVE_PATH = BASE_PATH / "models" / "garch_wf"
CKPT_DIR = MODEL_SAVE_PATH / "checkpoints"
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

WF_START = "2021-01-01"
WF_END = "2026-01-31"
WF_STEP_MONTHS = 3
LOOKBACK_DAYS = 252  # GARCH 피팅에 사용할 거래일
FORECAST_HORIZONS = [1, 3, 5]

# 윈도우 생성 (TFT/LightGBM WF와 동일)
windows = []
current = pd.Timestamp(WF_START)
end = pd.Timestamp(WF_END)
while current < end:
    test_end = current + pd.DateOffset(months=WF_STEP_MONTHS) - pd.Timedelta(days=1)
    if test_end > end: test_end = end
    train_end = current - pd.Timedelta(days=1)
    windows.append((str(train_end.date()), str(current.date()), str(test_end.date())))
    current += pd.DateOffset(months=WF_STEP_MONTHS)

print(f"Walk-Forward 윈도우: {len(windows)}개")

In [ ]:
# ===== 종목 목록 로드 =====
df_meta = pd.read_parquet(str(TFT_FEATURE_PATH), columns=["ticker"])
tickers = sorted(df_meta["ticker"].unique())
print(f"종목: {len(tickers)}개")

# OHLCV 전체 로드 (한 번만)
ohlcv_data = {}
for ticker in tickers:
    path = RAW_OHLCV_PATH / f"{ticker}.parquet"
    if path.exists():
        try:
            odf = pd.read_parquet(str(path))
            odf.index = pd.to_datetime(odf.index)
            ohlcv_data[ticker] = odf
        except: pass
print(f"OHLCV 로드: {len(ohlcv_data)} 종목")

In [ ]:
# ===== 단일 종목 GARCH 피팅 함수 =====
def fit_garch_single(ticker, ohlcv_df, forecast_date):
    """단일 종목 GARCH(1,1) 피팅 -> 변동성 예측 반환."""
    df = ohlcv_df.copy()
    df = df[df.index <= pd.Timestamp(forecast_date)]
    
    if "close" not in df.columns or len(df) < 60:
        return None
    
    returns = df["close"].pct_change(1).dropna() * 100
    returns = returns.tail(LOOKBACK_DAYS)
    
    if len(returns) < 60:
        return None
    
    try:
        model = arch_model(returns, vol="Garch", p=1, q=1, mean="Constant", rescale=False)
        result = model.fit(disp="off", show_warning=False)
        forecasts = result.forecast(horizon=max(FORECAST_HORIZONS))
        var_fc = forecasts.variance.iloc[-1]
        
        vols = {}
        for h in FORECAST_HORIZONS:
            col = f"h.{h}"
            if col in var_fc.index:
                vols[f"vol_{h}d"] = float(np.sqrt(var_fc[col]) * np.sqrt(252))
            else:
                vols[f"vol_{h}d"] = float("nan")
        
        # 리스크 플래그
        vol_5d = vols.get("vol_5d", float("nan"))
        vols["risk_flag"] = 1 if vol_5d > 30.0 else 0
        vols["ticker"] = ticker
        vols["date"] = forecast_date
        return vols
    except:
        return None

print("GARCH 피팅 함수 정의 완료")

In [ ]:
# ===== Walk-Forward 실행 =====
wf_results = []

for i, (train_end, test_start, test_end) in enumerate(windows):
    print(f"\n{'='*60}")
    print(f"  Window [{i:2d}/{len(windows)}] train_end={train_end}, test={test_start}~{test_end}")
    print(f"{'='*60}")
    
    ckpt_path = CKPT_DIR / f"window_{i:02d}_te_{train_end}.parquet"
    
    try:
        # 체크포인트 로드 or 피팅
        if ckpt_path.exists():
            print(f"  체크포인트 로드: {ckpt_path.name}")
            vol_df = pd.read_parquet(str(ckpt_path))
        else:
            print(f"  GARCH 피팅 중... ({len(ohlcv_data)} 종목)")
            
            # 테스트 기간의 각 거래일에 대해 GARCH 피팅
            # 실용적 접근: train_end 기준으로 한 번만 피팅 (3개월 동안 유효)
            vol_records = []
            success, fail = 0, 0
            
            for ticker in ohlcv_data:
                result = fit_garch_single(ticker, ohlcv_data[ticker], train_end)
                if result:
                    vol_records.append(result)
                    success += 1
                else:
                    fail += 1
            
            if vol_records:
                vol_df = pd.DataFrame(vol_records)
                # 체크포인트 저장
                vol_df.to_parquet(str(ckpt_path), index=False)
                print(f"  체크포인트 저장: {ckpt_path.name} (성공={success}, 실패={fail})")
            else:
                print(f"  SKIP: 피팅 성공 종목 없음")
                wf_results.append({"window": i, "train_end": train_end, "error": "피팅 실패"})
                continue
        
        # 결과 요약
        mean_vol_5d = vol_df["vol_5d"].mean()
        high_risk_pct = vol_df["risk_flag"].mean() * 100
        
        result = {"window": i, "train_end": train_end, "test_start": test_start,
                  "test_end": test_end, "n_tickers": len(vol_df),
                  "mean_vol_5d": round(mean_vol_5d, 2),
                  "high_risk_pct": round(high_risk_pct, 1),
                  "ckpt": str(ckpt_path)}
        wf_results.append(result)
        print(f"  종목: {len(vol_df)} | 평균 vol_5d: {mean_vol_5d:.1f}% | 고위험: {high_risk_pct:.1f}%")
        
    except Exception as e:
        print(f"  ERROR: {e}")
        wf_results.append({"window": i, "train_end": train_end, "error": str(e)})

print(f"\nWalk-Forward 완료: {len(wf_results)}개 윈도우")
print(f"체크포인트: {CKPT_DIR}")

In [ ]:
# ===== 결과 요약 =====
valid = [r for r in wf_results if "error" not in r]
if valid:
    print("="*70)
    print("  GARCH Walk-Forward 결과")
    print("="*70)
    print(f"  윈도우: {len(valid)}개 / {len(wf_results)}개")
    print()
    print(f"  {'윈도우':>6} {'기간':>24} {'종목수':>6} {'평균vol_5d':>10} {'고위험':>8}")
    print(f"  {'-'*60}")
    for r in valid:
        print(f"  [{r['window']:2d}] {r['test_start']}~{r['test_end']}  {r['n_tickers']:>5}  {r['mean_vol_5d']:>8.1f}%  {r['high_risk_pct']:>6.1f}%")
    print("="*70)
    
    # 시간에 따른 변동성 추이
    vols = [r["mean_vol_5d"] for r in valid]
    print(f"\n  변동성 추이: min={min(vols):.1f}%, max={max(vols):.1f}%, mean={np.mean(vols):.1f}%")

In [ ]:
# ===== 저장 =====
json.dump(wf_results, open(str(MODEL_SAVE_PATH / f"wf_results_{datetime.now().strftime('%Y%m%d')}.json"), "w"),
          ensure_ascii=False, indent=2)
print(f"결과 저장: {MODEL_SAVE_PATH}")

## GARCH Walk-Forward (체크포인트)\n\n- 각 윈도우: train_end 기준으로 전 종목 GARCH(1,1) 피팅\n- 변동성 예측 (1d/3d/5d) + 리스크 플래그 저장\n- 체크포인트: 윈도우별 예측 DataFrame (parquet)\n\n### 앙상블에서의 활용\n- `risk_flag=1`인 종목: 매수 제외 또는 포지션 축소\n- `vol_5d`: 포지션 사이징에 활용 (변동성 역비례 배분)\n- 시장 전체 변동성: 매수 타이밍 조절\n\n저장 위치: `models/garch_wf/checkpoints/window_XX_te_YYYY-MM-DD.parquet`